In [0]:
%pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 63.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("folder_id", "19WHz3NGVyd4c7r56dRPIXQeDYwu8o_Nn")
dbutils.widgets.text("catalog_name", "resume")
dbutils.widgets.text("env", "")
catalog_name=dbutils.widgets.get("catalog_name")
folder_id=dbutils.widgets.get("folder_id")
env = dbutils.widgets.get("env")

In [0]:
catalog_name

'resume'

In [0]:

SERVICE_ACCOUNT_FILE = f"/Volumes/{catalog_name}/staging/metadata/resume_{env}.json"

In [0]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

folder_id = "your_folder_id_here"  # paste your folder ID here

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)
service = build("drive", "v3", credentials=creds)


## Incremental Load Strategy

This notebook implements incremental loading from Google Drive using file metadata:

### How it works:
1. **Metadata Table**: Tracks each file's `file_id`, `modified_time`, and `processed_time`
2. **Change Detection**: Compares Google Drive's `modifiedTime` with our metadata table
3. **Selective Download**: Only downloads files that are:
   - New (not in metadata table)
   - Modified (modifiedTime > last processed time)

### Benefits:
- Faster execution (skip unchanged files)
- Reduced API calls to Google Drive
- Lower storage I/O
- Audit trail of file processing history

### Metadata Table Schema:
```
file_id         : Unique Google Drive file ID
file_name       : Name of the file
file_path       : Local path in UC Volume
mime_type       : File type (PDF, DOCX, etc.)
modified_time   : Last modified time from Google Drive
processed_time  : When we last downloaded it
file_size       : File size in bytes
```

### Force Full Reload:
To reprocess all files, clear the metadata table:
```python
spark.sql(f"TRUNCATE TABLE {metadata_table}")
```

## Performance Tuning for Variable Data Volumes

This notebook adapts to different data volumes using query parameters:

### Parameters:

**1. `max_workers`** - Maximum parallel download threads
- **1**: Sequential (testing/debugging)
- **3-5**: Moderate loads (hundreds of files)
- **10-15**: Large loads (thousands of files)
- **20**: Very large loads (millions of records)

**2. `processing_mode`** - Processing strategy
- **auto**: Automatically scales workers based on file count (recommended)
- **small_batch**: Forces minimal parallelization (100s of records)
- **large_batch**: Forces maximum parallelization (millions of records)

### Auto-Scaling Logic:

When `processing_mode = "auto"`, the system automatically adjusts:

| File Count | Workers Used | Use Case |
|------------|--------------|----------|
| 1-5        | 1            | Very small batches |
| 6-20       | 3            | Small batches (hundreds) |
| 21-50      | 5            | Medium batches (thousands) |
| 51-100     | 10           | Large batches (hundreds of thousands) |
| 100+       | max_workers  | Very large batches (millions) |

### Recommendations:

**For Small Loads (100s of records):**
```python
max_workers = 3
processing_mode = "small_batch"
```

**For Large Loads (Millions of records):**
```python
max_workers = 15-20
processing_mode = "large_batch"
```

**For Variable Loads:**
```python
max_workers = 10
processing_mode = "auto"  # Let the system decide
```

### Notes:
- Higher worker counts may hit Google Drive API rate limits
- Auto mode provides the best balance for unpredictable workloads
- Sequential mode (workers=1) is useful for debugging

In [0]:
folder_id = "19WHz3NGVyd4c7r56dRPIXQeDYwu8o_Nn"
files = service.files().list(
    q=f"'{folder_id}' in parents and trashed=false",
    fields="files(id, name, mimeType)"
).execute().get("files", [])

for f in files:
    print(f['name'], "-", f['mimeType'])

Resume_Data_PDF - application/vnd.google-apps.folder
Company_JD_s - application/vnd.google-apps.folder


In [0]:
import io
import os
from googleapiclient.http import MediaIoBaseDownload

UC_VOLUME_PATH = "/Volumes/resume_batch3/staging/source_files"  # update this
EXPORT_MAP = {
    "application/vnd.google-apps.document":     ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".docx"),
    "application/vnd.google-apps.spreadsheet":  ("application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": ("application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
    "application/vnd.google-apps.drawing":      ("image/png", ".png"),
}

def download_folder(folder_id, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    items = service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType)"
    ).execute().get("files", [])

    for f in items:
        file_id   = f["id"]
        file_name = f["name"]
        mime_type = f["mimeType"]

        if mime_type == "application/vnd.google-apps.folder":
            print(f"Entering subfolder: {file_name}")
            download_folder(file_id, f"{dest_dir}/{file_name}")

        elif mime_type in EXPORT_MAP:
            export_mime, ext = EXPORT_MAP[mime_type]
            request = service.files().export_media(fileId=file_id, mimeType=export_mime)
            if not file_name.endswith(ext):
                file_name += ext
            _save(request, f"{dest_dir}/{file_name}")

        elif mime_type.startswith("application/vnd.google-apps."):
            print(f"  Skipped (unsupported type): {file_name} [{mime_type}]")

        else:
            request = service.files().get_media(fileId=file_id)
            _save(request, f"{dest_dir}/{file_name}")

def _save(request, dest_path):
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    with open(dest_path, "wb") as out:
        out.write(buffer.read())
    print(f"  Copied → {dest_path}")

download_folder(folder_id, UC_VOLUME_PATH)

Entering subfolder: Resume_Data_PDF
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Vikram Singh - Senior Data Engineer Resume.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Chris Anderson - Performance QA Engineer Resume.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Sara_Johnson_Java.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Emily Davis - QA Engineer Resume.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Sagar Prajapati - Resume.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Rahul Gupta Senior Java Developer.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Priya_Sharma_Java.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Rajesh Kumar - Java Developer Resume.pdf
  Copied → /Volumes/resume_batch3/staging/source_files/Resume_Data_PDF/Priya Sharma Java Developer.pdf
  Cop

In [0]:
# Create metadata table for incremental load tracking
metadata_table = f"{catalog_name}.staging.file_processing_metadata"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
    file_id STRING,
    file_name STRING,
    file_path STRING,
    mime_type STRING,
    modified_time TIMESTAMP,
    processed_time TIMESTAMP,
    file_size BIGINT
) USING DELTA
""")

print(f"Metadata table ready: {metadata_table}")

Metadata table ready: resume_batch3.staging.file_processing_metadata


In [0]:
import io
import os
from datetime import datetime, timezone
from googleapiclient.http import MediaIoBaseDownload
from pyspark.sql import Row

UC_VOLUME_PATH = f"/Volumes/{catalog_name}/staging/source_files"
EXPORT_MAP = {
    "application/vnd.google-apps.document":     ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".docx"),
    "application/vnd.google-apps.spreadsheet":  ("application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": ("application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
    "application/vnd.google-apps.drawing":      ("image/png", ".png"),
}

def get_existing_metadata():
    """Load existing metadata as a dictionary keyed by file_id"""
    try:
        df = spark.table(metadata_table)
        metadata_dict = {row.file_id: row.asDict() for row in df.collect()}
        return metadata_dict
    except:
        return {}

def needs_download(file_id, modified_time_str, existing_metadata):
    """Check if file needs to be downloaded based on modification time"""
    if file_id not in existing_metadata:
        return True  # New file
    
    # Parse Google Drive modifiedTime (ISO 8601 format) - timezone-aware
    modified_time = datetime.fromisoformat(modified_time_str.replace('Z', '+00:00'))
    
    # Get last processed time and make it timezone-aware if it's not
    last_processed = existing_metadata[file_id]['modified_time']
    if last_processed.tzinfo is None:
        last_processed = last_processed.replace(tzinfo=timezone.utc)
    
    # Download if file has been modified since last processing
    return modified_time > last_processed

def download_folder_incremental(folder_id, dest_dir, existing_metadata):
    """Recursively download only new or modified files"""
    os.makedirs(dest_dir, exist_ok=True)
    
    # Request file metadata including modifiedTime and size
    items = service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType, modifiedTime, size)"
    ).execute().get("files", [])
    
    processed_files = []
    
    for f in items:
        file_id = f["id"]
        file_name = f["name"]
        mime_type = f["mimeType"]
        modified_time = f.get("modifiedTime")
        file_size = int(f.get("size", 0)) if f.get("size") else 0
        
        # Handle folders recursively
        if mime_type == "application/vnd.google-apps.folder":
            print(f"Entering subfolder: {file_name}")
            subfolder_files = download_folder_incremental(
                file_id, 
                f"{dest_dir}/{file_name}", 
                existing_metadata
            )
            processed_files.extend(subfolder_files)
            continue
        
        # Check if file needs downloading
        if not needs_download(file_id, modified_time, existing_metadata):
            print(f"  Skipped (unchanged): {file_name}")
            continue
        
        # Download logic
        file_path = None
        if mime_type in EXPORT_MAP:
            export_mime, ext = EXPORT_MAP[mime_type]
            request = service.files().export_media(fileId=file_id, mimeType=export_mime)
            if not file_name.endswith(ext):
                file_name += ext
            file_path = f"{dest_dir}/{file_name}"
            _save(request, file_path)
            
        elif mime_type.startswith("application/vnd.google-apps."):
            print(f"  Skipped (unsupported type): {file_name} [{mime_type}]")
            continue
            
        else:
            request = service.files().get_media(fileId=file_id)
            file_path = f"{dest_dir}/{file_name}"
            _save(request, file_path)
        
        # Record metadata for this file
        if file_path:
            processed_files.append({
                "file_id": file_id,
                "file_name": file_name,
                "file_path": file_path,
                "mime_type": mime_type,
                "modified_time": datetime.fromisoformat(modified_time.replace('Z', '+00:00')),
                "processed_time": datetime.now(timezone.utc),
                "file_size": file_size
            })
    
    return processed_files

def _save(request, dest_path):
    """Save file to destination"""
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    with open(dest_path, "wb") as out:
        out.write(buffer.read())
    print(f"  Downloaded → {dest_path}")

# Execute incremental download
print(f"Starting incremental load from folder: {folder_id}")
existing_metadata = get_existing_metadata()
print(f"Found {len(existing_metadata)} files in metadata table")

processed_files = download_folder_incremental(folder_id, UC_VOLUME_PATH, existing_metadata)
print(f"\nProcessed {len(processed_files)} new/modified files")

Starting incremental load from folder: 19WHz3NGVyd4c7r56dRPIXQeDYwu8o_Nn
Found 21 files in metadata table
Entering subfolder: Resume_Data_PDF
  Skipped (unchanged): Vikram Singh - Senior Data Engineer Resume.pdf
  Skipped (unchanged): Chris Anderson - Performance QA Engineer Resume.pdf
  Skipped (unchanged): Sara_Johnson_Java.pdf
  Skipped (unchanged): Emily Davis - QA Engineer Resume.pdf
  Skipped (unchanged): Sagar Prajapati - Resume.pdf
  Skipped (unchanged): Rahul Gupta Senior Java Developer.pdf
  Skipped (unchanged): Priya_Sharma_Java.pdf
  Skipped (unchanged): Rajesh Kumar - Java Developer Resume.pdf
  Skipped (unchanged): Priya Sharma Java Developer.pdf
  Skipped (unchanged): Ananya Gupta - Java Developer Resume.pdf
  Skipped (unchanged): James Thompson - QA Automation Lead Resume.pdf
  Skipped (unchanged): Jessica Taylor - Product Manager Resume.pdf
  Skipped (unchanged): Michal_Java.pdf
  Skipped (unchanged): Meera Krishnan - Staff Data Engineer Resume.pdf
  Skipped (unchanged

In [0]:
# Update metadata table with newly processed files
if processed_files:
    # Convert to DataFrame
    processed_df = spark.createDataFrame([Row(**f) for f in processed_files])
    
    # Create temp view for merge
    processed_df.createOrReplaceTempView("new_files")
    
    # MERGE into metadata table
    spark.sql(f"""
    MERGE INTO {metadata_table} AS target
    USING new_files AS source
    ON target.file_id = source.file_id
    WHEN MATCHED THEN UPDATE SET
        target.file_name = source.file_name,
        target.file_path = source.file_path,
        target.mime_type = source.mime_type,
        target.modified_time = source.modified_time,
        target.processed_time = source.processed_time,
        target.file_size = source.file_size
    WHEN NOT MATCHED THEN INSERT (
        file_id, file_name, file_path, mime_type, 
        modified_time, processed_time, file_size
    ) VALUES (
        source.file_id, source.file_name, source.file_path, source.mime_type,
        source.modified_time, source.processed_time, source.file_size
    )
    """)
    
    print(f"✓ Metadata table updated with {len(processed_files)} file(s)")
    
    # Show summary
    display(spark.sql(f"""
        SELECT 
            COUNT(*) as total_files,
            MAX(modified_time) as latest_file_modified,
            MAX(processed_time) as last_run_time,
            SUM(file_size) as total_size_bytes,
            ROUND(SUM(file_size) / 1024 / 1024, 2) as total_size_mb
        FROM {metadata_table}
    """))
else:
    print("✓ No new or modified files to process")

✓ No new or modified files to process


In [0]:
#Query to view current metadata status
#Uncomment and run to see what files have been processed

display(spark.sql(f"""
     SELECT 
         file_name,
         mime_type,
         modified_time,
         processed_time,
         ROUND(file_size / 1024, 2) as size_kb,
         DATEDIFF(HOUR, modified_time, processed_time) as lag_hours
     FROM {metadata_table}
     ORDER BY processed_time DESC
     LIMIT 50
 """))

file_name,mime_type,modified_time,processed_time,size_kb,lag_hours
Data Engineer - Accenture.docx,application/vnd.openxmlformats-officedocument.wordprocessingml.document,2026-03-20T20:11:00.000Z,2026-03-24T16:30:49.514Z,7.45,92
QA Engineer_Accenture.docx,application/vnd.openxmlformats-officedocument.wordprocessingml.document,2026-03-20T20:11:00.000Z,2026-03-24T16:30:48.941Z,8.8,92
Java Developer_Accenture.docx,application/vnd.openxmlformats-officedocument.wordprocessingml.document,2026-03-20T20:11:00.000Z,2026-03-24T16:30:48.527Z,8.84,92
Data Engineer - Wipro.docx,application/vnd.openxmlformats-officedocument.wordprocessingml.document,2026-03-20T20:11:00.000Z,2026-03-24T16:30:48.098Z,7.46,92
Data Engineer - TCS.docx,application/vnd.openxmlformats-officedocument.wordprocessingml.document,2026-03-20T20:11:00.000Z,2026-03-24T16:30:47.537Z,7.56,92
Ryan O_Connor - Technical Product Manager Resume.pdf,application/pdf,2026-03-20T20:11:00.000Z,2026-03-24T16:30:46.672Z,175.13,92
Daniel Park - Data Engineer Resume.pdf,application/pdf,2026-03-20T20:11:00.000Z,2026-03-24T16:30:45.966Z,173.58,92
Meera Krishnan - Staff Data Engineer Resume.pdf,application/pdf,2026-03-20T20:11:00.000Z,2026-03-24T16:30:45.301Z,179.46,92
Michal_Java.pdf,application/pdf,2026-03-20T20:11:00.000Z,2026-03-24T16:30:44.010Z,141.01,92
Jessica Taylor - Product Manager Resume.pdf,application/pdf,2026-03-20T20:11:00.000Z,2026-03-24T16:30:43.084Z,177.96,92


In [0]:
import io
import os
from datetime import datetime, timezone
from googleapiclient.http import MediaIoBaseDownload
from pyspark.sql import Row
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

UC_VOLUME_PATH = f"/Volumes/{catalog_name}/staging/source_files"
EXPORT_MAP = {
    "application/vnd.google-apps.document":     ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".docx"),
    "application/vnd.google-apps.spreadsheet":  ("application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": ("application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
    "application/vnd.google-apps.drawing":      ("image/png", ".png"),
}

# Get parameters
max_workers_param = int(dbutils.widgets.get("max_workers"))
processing_mode = dbutils.widgets.get("processing_mode")

def determine_optimal_workers(file_count, mode, max_workers_param):
    """
    Determine optimal number of workers based on file count and processing mode.
    
    Args:
        file_count: Number of files to process
        mode: 'auto', 'small_batch', or 'large_batch'
        max_workers_param: Maximum workers from parameter
    
    Returns:
        Optimal number of workers
    """
    if mode == "small_batch":
        # For small batches (100s of records), use minimal workers
        return min(2, max_workers_param, file_count)
    
    elif mode == "large_batch":
        # For large batches (millions of records), use maximum workers
        return min(max_workers_param, file_count)
    
    else:  # auto mode
        # Adaptive scaling based on file count
        if file_count <= 5:
            workers = 1  # Sequential for very small loads
        elif file_count <= 200:
            workers = min(2, max_workers_param)  # Light parallelization
        elif file_count <= 500:
            workers = min(3, max_workers_param)  # Moderate parallelization
        elif file_count <= 1000:
            workers = min(5, max_workers_param)  # Heavy parallelization
        else:
            workers = max_workers_param  # Maximum for very large loads
        
        return min(workers, file_count)

def get_existing_metadata():
    """Load existing metadata as a dictionary keyed by file_id"""
    try:
        df = spark.table(metadata_table)
        metadata_dict = {row.file_id: row.asDict() for row in df.collect()}
        return metadata_dict
    except:
        return {}

def needs_download(file_id, modified_time_str, existing_metadata):
    """Check if file needs to be downloaded based on modification time"""
    if file_id not in existing_metadata:
        return True  # New file
    
    # Parse Google Drive modifiedTime (ISO 8601 format) - timezone-aware
    modified_time = datetime.fromisoformat(modified_time_str.replace('Z', '+00:00'))
    
    # Get last processed time and make it timezone-aware if it's not
    last_processed = existing_metadata[file_id]['modified_time']
    if last_processed.tzinfo is None:
        last_processed = last_processed.replace(tzinfo=timezone.utc)
    
    # Download if file has been modified since last processing
    return modified_time > last_processed

def download_single_file(file_info, dest_dir, existing_metadata):
    """Download a single file (called by worker threads)"""
    file_id = file_info["id"]
    file_name = file_info["name"]
    mime_type = file_info["mimeType"]
    modified_time = file_info.get("modifiedTime")
    file_size = int(file_info.get("size", 0)) if file_info.get("size") else 0
    
    # Check if file needs downloading
    if not needs_download(file_id, modified_time, existing_metadata):
        print(f"  Skipped (unchanged): {file_name}")
        return None
    
    # Download logic
    file_path = None
    try:
        if mime_type in EXPORT_MAP:
            export_mime, ext = EXPORT_MAP[mime_type]
            request = service.files().export_media(fileId=file_id, mimeType=export_mime)
            if not file_name.endswith(ext):
                file_name += ext
            file_path = f"{dest_dir}/{file_name}"
            _save(request, file_path)
            
        elif mime_type.startswith("application/vnd.google-apps."):
            print(f"  Skipped (unsupported type): {file_name} [{mime_type}]")
            return None
            
        else:
            request = service.files().get_media(fileId=file_id)
            file_path = f"{dest_dir}/{file_name}"
            _save(request, file_path)
        
        # Return metadata for this file
        if file_path:
            return {
                "file_id": file_id,
                "file_name": file_name,
                "file_path": file_path,
                "mime_type": mime_type,
                "modified_time": datetime.fromisoformat(modified_time.replace('Z', '+00:00')),
                "processed_time": datetime.now(timezone.utc),
                "file_size": file_size
            }
    except Exception as e:
        print(f"  Error downloading {file_name}: {str(e)}")
        return None

def download_folder_incremental(folder_id, dest_dir, existing_metadata):
    """Recursively download only new or modified files with adaptive parallel processing"""
    os.makedirs(dest_dir, exist_ok=True)
    
    # Request file metadata including modifiedTime and size
    items = service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType, modifiedTime, size)"
    ).execute().get("files", [])
    
    processed_files = []
    files_to_download = []
    
    for f in items:
        mime_type = f["mimeType"]
        
        # Handle folders recursively (not parallelized - folders first)
        if mime_type == "application/vnd.google-apps.folder":
            file_name = f["name"]
            print(f"Entering subfolder: {file_name}")
            subfolder_files = download_folder_incremental(
                f["id"], 
                f"{dest_dir}/{file_name}", 
                existing_metadata
            )
            processed_files.extend(subfolder_files)
        else:
            # Queue files for parallel download
            files_to_download.append(f)
    
    # Determine optimal workers for this batch
    optimal_workers = determine_optimal_workers(
        len(files_to_download), 
        processing_mode, 
        max_workers_param
    )
    
    # Parallel download of files in current folder
    if files_to_download:
        print(f"Processing {len(files_to_download)} file(s) with {optimal_workers} worker(s) [{processing_mode} mode]...")
        
        if optimal_workers == 1:
            # Sequential processing for very small batches
            for file_info in files_to_download:
                result = download_single_file(file_info, dest_dir, existing_metadata)
                if result:
                    processed_files.append(result)
        else:
            # Parallel processing
            with ThreadPoolExecutor(max_workers=optimal_workers) as executor:
                # Submit all download tasks
                future_to_file = {
                    executor.submit(download_single_file, file_info, dest_dir, existing_metadata): file_info
                    for file_info in files_to_download
                }
                
                # Collect results as they complete
                for future in as_completed(future_to_file):
                    result = future.result()
                    if result:
                        processed_files.append(result)
    
    return processed_files

def _save(request, dest_path):
    """Save file to destination"""
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    with open(dest_path, "wb") as out:
        out.write(buffer.read())
    print(f"  Downloaded → {dest_path}")

# Execute incremental download with adaptive multithreading
print(f"Starting incremental load from folder: {folder_id}")
print(f"Configuration: max_workers={max_workers_param}, mode={processing_mode}")
existing_metadata = get_existing_metadata()
print(f"Found {len(existing_metadata)} files in metadata table\n")

processed_files = download_folder_incremental(folder_id, UC_VOLUME_PATH, existing_metadata)
print(f"\n✓ Processed {len(processed_files)} new/modified files")

Starting incremental load from folder: 19WHz3NGVyd4c7r56dRPIXQeDYwu8o_Nn
Configuration: max_workers=10, mode=auto
Found 21 files in metadata table

Entering subfolder: Resume_Data_PDF
Processing 16 file(s) with 2 worker(s) [auto mode]...
  Skipped (unchanged): Vikram Singh - Senior Data Engineer Resume.pdf
  Skipped (unchanged): Chris Anderson - Performance QA Engineer Resume.pdf
  Skipped (unchanged): Sara_Johnson_Java.pdf
  Skipped (unchanged): Emily Davis - QA Engineer Resume.pdf
  Skipped (unchanged): Sagar Prajapati - Resume.pdf
  Skipped (unchanged): Rahul Gupta Senior Java Developer.pdf
  Skipped (unchanged): Priya_Sharma_Java.pdf
  Skipped (unchanged): Rajesh Kumar - Java Developer Resume.pdf
  Skipped (unchanged): Priya Sharma Java Developer.pdf
  Skipped (unchanged): Ananya Gupta - Java Developer Resume.pdf
  Skipped (unchanged): James Thompson - QA Automation Lead Resume.pdf
  Skipped (unchanged): Jessica Taylor - Product Manager Resume.pdf
  Skipped (unchanged): Michal_Java